<a href="https://colab.research.google.com/github/Kwadwo-Awuah/Lab-4/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [3]:
from google.colab import userdata
API_KEY = userdata.get('AP_key')   # use the secret name you chose

# Then instantiate the client (remove the base_url line if using OpenAI itself)
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # keep this for Groq
)
MODEL = "llama-3.3-70b-versatile"   # Groq's Llama variant
print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [4]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content
answer = ask_llm("What is the capital of Ghana?")
print("Answer: ", answer)

# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

Answer:  The capital of Ghana is Accra.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** The system message sets the persona of the assistant, its giving a description before it hears the question. The user is the actual question

2. A token is about 3/4 of a word.Providers bill per toen because it computs the cost scales with the number of inputs  + output

### Part 1.2 — Temperature: the randomness dial

In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a footballer in Ghana"
print("Temperature used = 0.0")
for i in range(5):
  answer = ask_llm(test_question,temperature =0.0, max_tokens = 50)
  print ("Run ",i ," ", answer)

print("temperature used = 1.2")
print("====")
for i in range(5):
  answer = ask_llm(test_question, temperature = 1.2, max_tokens = 50)
  print("Run ", i, " ", answer)


# TODO: Print all 10 answers, grouped by temperature.

Temperature used = 0.0
Run  0   How about "Kofi Owusu"? Kofi is a popular Ghanaian name that means "born on a Friday," and Owusu is a common Ghanaian surname. This name has a strong and athletic sound to it, fitting for a footballer
Run  1   How about "Kofi Owusu"? Kofi is a popular Ghanaian name that means "born on a Friday," and Owusu is a common surname in Ghana. This name has a strong and athletic sound to it, fitting for a footballer
Run  2   How about "Kofi Owusu"? Kofi is a popular Ghanaian name that means "born on a Friday," and Owusu is a common surname in Ghana. This name has a strong and athletic sound to it, fitting for a footballer
Run  3   How about "Kofi Owusu"? Kofi is a popular Ghanaian name that means "born on a Friday," and Owusu is a common surname in Ghana. The name has a strong and athletic sound to it, fitting for a footballer
Run  4   How about "Kofi Owusu"? Kofi is a popular Ghanaian name that means "born on a Friday," and Owusu is a common surname in Ghana. Th

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0, the outputs are all identical across all runs, it gave consistent suggestions. At temperature 1.2 the outputs vary. For the loan support system temperature 0 is more appropriate because we need consistent outputs. High temperature brings in randomness which could lead to a different decisions on the same applications.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [7]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically
print("Naive Prompt")
v1_user_prompt = "Summarize this loan application: \n{letter}"

for letter_id in ["L002",'L006']:
  letter_text = LETTERS[letter_id]
  prompt = v1_user_prompt.format(letter=letter_text)
  summary =ask_llm(prompt, system_prompt="You are a helpful assistant",temperature = 0.0)
  print("\n---", letter_id, "(V1) ---")
  print(summary)
print(" ")
print("Not naive prompt")
v2_system_prompt = "You are an assistant to a microfinance loan officer.Your task is to summarise loan application letters.Be factual and neutral. Do NOT add any information that is not present in the letter.Use exactly 3–4 sentences. Focus on the applicant, requested amount, business, repayment plan, and any collateral/guarantor. "
v2_user_prompt = "Summarise this loan application: \n{letter}"

for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]
  prompt = v2_user_prompt.format(letter=letter_text)
  summary = ask_llm(prompt,system_prompt=v2_system_prompt,temperature = 0.0)
  print(f"\n--- {letter_id} (V2) ---")
  print(summary)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

Naive Prompt

--- L002 (V1) ---
Here is a summary of the loan application:

* Applicant: Kwame Boateng, a commercial driver in Kumasi
* Loan amount: GHS 25,000
* Purpose: To repair his trotro engine and settle personal debts
* Repayment plan: No specific plan, but promises to pay back when business picks up after the festive season
* Collateral: None available at the moment

--- L006 (V1) ---
Here's a summary of Kofi's loan application:

* Amount requested: GHS 50,000
* Proposed businesses: car washing, provision shop, and importing phones from Dubai
* Applicant's age: 22
* Repayment plan: 1 year, relying on the success of the businesses
* Collateral: None, but Kofi claims to be trustworthy
* Experience: No prior experience in any of the proposed businesses, but friends consider him "business-minded"
 
Not naive prompt

--- L002 (V2) ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He expects h

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** It made things sound skeptical and invented a claim of trustworthiness  and it was also interpretive like saying "i can pay back whenever the money comes"
V2 fixes this by removing such language and making the summary concise in 3 or 4 sentences it also removes anything that sounds subjective

2. This is because these details could mislead the loan officer into making the wrong decisons, even adding a little details could completely change the applicants request. Failure mode is called hallucination and here the model creates information not found in the input

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [8]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
import json
import pandas as pd

Extract_system = "You are an AI that extracts structured data from loan application lettersReturn ONLY a JSON object with these keys:- applicant_name (string)- amount_ghs (number)- purpose (string)- monthly_profit_ghs (number or null)- has_collateral_or_guarantor (boolean)- repayment_months (number or null) If a field is not mentioned in the letter, use null. Do not guess or infer.Here is an example:Letter: My name is Ama. I need 5000 cedis to buy sewing machines. I make 600 a month. I can repay in 10 months. My brother will guarantee.Output: {applicant_name: Ama, amount_ghs: 5000,purpose: buy sewing machines, monthly_profit_ghs: 600, has_collateral_or_guarantor: true, repayment_months: 10}Now extract from the following letter. Return ONLY the JSON object. Do not include any other text."
def extract_fields(letter_text):
  response = ask_llm(letter_text, system_prompt = Extract_system, temperature= 0.0)
  response = response.strip()
  if response.startswith("```json"):
    response = response[7:]
  if response.startswith("```"):
    response = response[3:]
  if response.endswith("```"):
    response = response[:-3]
  response = response.strip()
  try:
     return json.loads(response)
  except:
    print("Failed to parse")
    print(response)
    return None

results = {}
for letter_id, letter_text in LETTERS.items():
    print(f"Extracting {letter_id}...")
    data = extract_fields(letter_text)
    results[letter_id] = data

# Create DataFrame
df = pd.DataFrame(results).T
print("\n" + "=" * 60)
print("EXTRACTED DATA")
print("=" * 60)
display(df)

#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

Extracting L001...
Extracting L002...
Extracting L003...
Extracting L004...
Extracting L005...
Extracting L006...

EXTRACTED DATA


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900,True,20
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,None,False,None
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800,True,15
L004,Yaw Owusu,12000,for feed and 500 new layers,1500,True,18
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,None,True,16
L006,Kofi,50000,"start a car washing business, a provision shop...",None,False,12


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [14]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
brief_system = "You are a decision support tool for a microfinance loan officer Your task is to analyze loan applications and provide a balanced, factual brief.For each application, output EXACTLY these 4 sections:STRENGTHS: bullet points of positive factors (grounded in the letter)2. RISKS / RED FLAGS: bullet points of concerns (grounded in the letter)3. MISSING INFORMATION: what the officer should ask the applicant4. SUGGESTED NEXT STEP: one of these options only:- Invite for interview- Request additional documents- Flag for senior review- Monitor and follow up IMPORTANT: - Do NOT say approve or reject – the final decision is made by a human officer.- Base everything ONLY on what's in the letter and extracted data.- Be neutral and objective. "
def generate_brief(letter_id,letter_text,extract_fields):
  extracted_str = json.dumps(extract_fields, indent = 2)
  user_prompt = f"""Loan Application: {letter_id}

ORIGINAL LETTER:
{letter_text}

EXTRACTED DATA:
{extracted_str}

Please provide a decision-support brief with the 4 sections as instructed."""

  brief = ask_llm(user_prompt, system_prompt=brief_system, temperature=0.0)
  return brief


print("=" * 60)
print("DECISION-SUPPORT BRIEFS")
print("=" * 60)

for letter_id, letter_text in LETTERS.items():
    extracted_fields = results.get(letter_id)

    if extracted_fields is None:
        print(f"\n--- {letter_id} ---")
        print("No extracted data available – skipping")
        continue

    print(f"\n--- {letter_id} ---")
    brief = generate_brief(letter_id, letter_text, extracted_fields)
    print(brief)


#    1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

DECISION-SUPPORT BRIEFS

--- L001 ---
## STRENGTHS:
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, which indicates a consistent income stream.
* Akosua has demonstrated a savings habit through the susu scheme, accumulating GHS 2,500 over two years without missing any contributions.
* She has a guarantor, her sister, who is a teacher, potentially providing an added layer of security for the loan.
* The applicant has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods, potentially increasing her business income.

## RISKS / RED FLAGS:
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, which might pose a risk if the business expansion does not generate enough additional income to cover the loan repayments.
* The repayment plan of GHS 450 over 20 months is relatively aggressive and might strai

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** Yes the system identified the strong factors and the red flag in the other one, it didnt approve/reject it gave clear information tho

2. Automated loan decisions can discriminate against certain groups of people, keeping a human in loop keeps the fair element present and the ability to appeal

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [15]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
# Part 4.1 – Extraction Accuracy

# GOLD labels (from Section 2)
GOLD = {
    "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000, "purpose": "buy deep freezer / expand into frozen foods",
             "monthly_profit_ghs": 900, "has_collateral_or_guarantor": True, "repayment_months": 20},
    "L003": {"applicant_name": "Efua Darko", "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
             "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True, "repayment_months": 15},
    "L006": {"applicant_name": "Kofi", "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
             "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

# Compare extracted vs gold
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]
accuracy = {}

print("=" * 60)
print("EXTRACTION ACCURACY VS GOLD STANDARD")
print("=" * 60)

for letter_id in ["L001", "L003", "L006"]:
    extracted = results.get(letter_id)
    gold = GOLD.get(letter_id)

    if extracted is None:
        print(f"{letter_id}: No extraction data available")
        continue

    print(f"\n--- {letter_id} ---")
    correct = 0
    total = len(fields)

    for field in fields:
        extracted_val = extracted.get(field)
        gold_val = gold.get(field)

        # Compare (case-insensitive for strings)
        if isinstance(extracted_val, str) and isinstance(gold_val, str):
            match = extracted_val.lower().strip() == gold_val.lower().strip()
        else:
            match = extracted_val == gold_val

        if match:
            correct += 1
            status = " MATCH"
        else:
            status = f" MISMATCH (extracted: {extracted_val}, gold: {gold_val})"

        print(f"  {field}: {status}")

    accuracy[letter_id] = correct / total
    print(f"  Accuracy: {correct}/{total} = {accuracy[letter_id]*100:.0f}%")

# Summary
print("\n" + "=" * 60)
print("SUMMARY ACCURACY")
print("=" * 60)
for letter_id, acc in accuracy.items():
    print(f"{letter_id}: {acc*100:.0f}%")
overall = sum(accuracy.values()) / len(accuracy)
print(f"Overall: {overall*100:.0f}%")
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

EXTRACTION ACCURACY VS GOLD STANDARD

--- L001 ---
  applicant_name:  MATCH
  amount_ghs:  MATCH
  purpose:  MISMATCH (extracted: buy a deep freezer and expand into frozen foods, gold: buy deep freezer / expand into frozen foods)
  monthly_profit_ghs:  MATCH
  has_collateral_or_guarantor:  MATCH
  repayment_months:  MATCH
  Accuracy: 5/6 = 83%

--- L003 ---
  applicant_name:  MATCH
  amount_ghs:  MATCH
  purpose:  MISMATCH (extracted: purchase two industrial sewing machines and fabric stock, gold: industrial sewing machines and fabric stock)
  monthly_profit_ghs:  MATCH
  has_collateral_or_guarantor:  MATCH
  repayment_months:  MATCH
  Accuracy: 5/6 = 83%

--- L006 ---
  applicant_name:  MATCH
  amount_ghs:  MATCH
  purpose:  MISMATCH (extracted: start a car washing business, a provision shop, and also import phones from Dubai, gold: car wash, provision shop, phone imports)
  monthly_profit_ghs:  MATCH
  has_collateral_or_guarantor:  MATCH
  repayment_months:  MATCH
  Accuracy: 5/6 = 8

### Part 4.2 — Reliability: is the system consistent?

In [20]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
# Part 4.2 – Reliability (consistency across runs)

import json
from collections import Counter

letter_to_test = "L004"
letter_text = LETTERS[letter_to_test]

results_temp0 = []
results_temp1 = []

# Run 5 times at temperature=0.0
print("=" * 60)
print("TEMPERATURE = 0.0")
print("=" * 60)
for i in range(5):
    data = extract_fields(letter_text)  # our function uses temp=0.0 by default
    results_temp0.append(data)
    print(f"Run {i+1}: {data}")

# Run 5 times at temperature=1.0
print("\n" + "=" * 60)
print("TEMPERATURE = 1.0")
print("=" * 60)
# We need to modify extract_fields to accept temperature, or call ask_llm directly.
# Let's create a temporary extraction function that accepts temperature.
def extract_fields_temp(letter, temp):
    response = ask_llm(letter, system_prompt=Extract_system, temperature=temp)
    # Clean and parse
    response = response.strip()
    if response.startswith("```json"):
        response = response[7:]
    if response.startswith("```"):
        response = response[3:]
    if response.endswith("```"):
        response = response[:-3]
    response = response.strip()
    try:
        return json.loads(response)
    except:
        return None

for i in range(5):
    data = extract_fields_temp(letter_text, temp=1.0)
    results_temp1.append(data)
    print(f"Run {i+1}: {data}")

# Analyze consistency
def analyze(results, temp_label):
    valid = [r for r in results if r is not None]
    print(f"\n{temp_label}:")
    print(f"  Valid JSON: {len(valid)} out of {len(results)}")
    if valid:
        # Count unique JSON strings (sorted keys)
        unique = set()
        for r in valid:
            # Sort keys to ensure consistent ordering
            unique.add(json.dumps(r, sort_keys=True))
        print(f"  Unique outputs: {len(unique)}")
        if len(unique) == 1:
            print("  → All valid outputs are IDENTICAL (consistent).")
        else:
            print("  → Outputs vary across runs.")
    else:
        print("  No valid JSON returned.")

analyze(results_temp0, "Temperature 0.0")
analyze(results_temp1, "Temperature 1.0")
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

TEMPERATURE = 0.0
Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

TEMPERATURE = 1.0
Run 1: {

### Part 4.3 — Hallucination probing

In [22]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
# Part 4.2 – Reliability (consistency across runs)

# Part 4.3 – Hallucination Probing

print("=" * 60)
print("HALLUCINATION PROBING TESTS")
print("=" * 60)

# ============================================
# TEST 1: Ask about a detail NOT in the letter
# ============================================
print("\n" + "=" * 60)
print("TEST 1: Asking about missing detail")
print("=" * 60)

letter_l001 = LETTERS["L001"]

# Ask about something NOT in the letter
test_prompt_1 = f"""Summarize this loan application:

{letter_l001}

Also, what is the applicant's credit score?"""

print("PROMPT:")
print(test_prompt_1)
print("\nRESPONSE:")
response_1 = ask_llm(test_prompt_1, system_prompt="You are a helpful assistant.", temperature=0.0)
print(response_1)

# Analyze
if "credit score" in response_1.lower() or "score" in response_1.lower():
    if "not" in response_1.lower() or "doesn't" in response_1.lower() or "not provided" in response_1.lower() or "no" in response_1.lower():
        print("\n PASS: Model correctly stated credit score is not provided.")
    else:
        print("\n FAIL: Model may have invented a credit score.")
else:
    print("\n UNCLEAR: Model didn't mention credit score at all.")


# ============================================
# TEST 2: Extract from irrelevant text
# ============================================
print("\n" + "=" * 60)
print("TEST 2: Extracting from irrelevant text")
print("=" * 60)

weather_text = """Today's weather in Accra is sunny with a high of 32°C.
Humidity is at 70% with light winds from the southwest.
There is a 10% chance of rain in the afternoon.
The UV index is high, so wear sunscreen if going outside."""

print("INPUT TEXT (Weather Report):")
print(weather_text)
print("\nEXTRACTED DATA:")
extracted_weather = extract_fields(weather_text)
print(extracted_weather)

# Analyze
if extracted_weather is None:
    print("\n PASS: Extract function returned None (handled gracefully).")
elif all(v is None for v in extracted_weather.values()):
    print("\n PASS: All fields are null (model didn't fabricate data).")
else:
    # Check if any fields have real-looking values
    fake_fields = [k for k, v in extracted_weather.items() if v is not None]
    if fake_fields:
        print(f"\n FAIL: Model fabricated data for: {fake_fields}")
        print(f"Extracted: {extracted_weather}")
    else:
        print("\n PASS: No fabricated data.")


# ============================================
# Summary
# ============================================
print("\n" + "=" * 60)
print("HALLUCINATION SUMMARY")
print("=" * 60)
print("Test 1 (credit score): The model should admit the information is not available.")
print("Test 2 (weather report): The model should return all nulls or handle gracefully.")
print("\nMark each as PASS or FAIL based on your outputs above.")
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

HALLUCINATION PROBING TESTS

TEST 1: Asking about missing detail
PROMPT:
Summarize this loan application:

Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.

Also, what is the applicant's credit score?

RESPONSE:
Here's a summary of the loan application:

Akosua Mensah, a 12-year vendor at Makola Market, is applying for a GHS 8,000 loan to expand her business into frozen foods by purchasing a deep freezer. She has a monthly profit of GHS 900 and has saved GHS 2,500 through the susu scheme over two years. She proposes to repay the loan at GHS 450 

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** The hardest was the purpose for letter 6 it had to capture all the three ventures in one string

2. for temp 0 all 5 runs were identical to the json output

for temp 1
some runs had invalid json it couldnt be parsed
its not reliable for production

the temp 0 must always be used for extraction and decision making tasks in production

3. Yes it correctly admitted the information was absent

yes it returned all the null instead of fabricating the applicant


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** The people who could be unfairly harmed would
1. Applicants with low literacy, when they are applying it might sound less detailed because of the their low literacy, even if the business plan is sound

they could be denied the loans they deserve and this could lead to them losing livelihood opportunities

2. Security, if the customer is breached all the data is completely exposed

Legal liabilities

what to check before deployment
 Data protection laws
 Encryption
 Consent

 3. Mandatory human review to keep every thing fair and prevent automatic bias

 Automated flagging when are application has missing critical fields it must get automatically flagged

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.